In [1]:
# Внимание!!! Важно, что бы файлы с данными и исполняемый файл находились в одной папке, 
# тогда пути к тестовым и тренировочным наборам будут содержать только имена файлов.
# 
# В пути к тренировочным и тестовым данным запрежается использовать абсалютную адресацию, 
# то есть адресацию, в которой присутствуют имена папок. Путь должен содержать только имя файла.
#
# Напоминание: под моделью машинного обучения понимаются все действия с исходными данными, 
# которые необходимо произвести, что бы сопоставить признаки целевому значению.

### Область работы 1 (библиотеки)

In [2]:
# Данный блок в области 1 выполняется преподавателем
# 
# данный блок предназначен только для подключения необходимых библиотек
# запрещается подключать библиотеки в других блоках
# запрещается скрывать предупреждения системы
# установка дополнительных библиотек размещается прямо здесь (обязательно закоментированы)
# pip install

In [3]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sklearn
import category_encoders as ce

from IPython.display import display
from tqdm.auto import tqdm
from tqdm_joblib import tqdm_joblib

from sklearn.datasets import load_iris

from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.compose import ColumnTransformer, make_column_transformer, TransformedTargetRegressor

from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import MinMaxScaler
from sklearn.preprocessing import RobustScaler
from sklearn.preprocessing import Normalizer
from sklearn.preprocessing import PolynomialFeatures
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import QuantileTransformer
from sklearn.preprocessing import PowerTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.preprocessing import KBinsDiscretizer

from sklearn.impute import SimpleImputer

from sklearn.model_selection import train_test_split
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import StratifiedShuffleSplit
from sklearn.model_selection import ParameterGrid
from sklearn.model_selection import TunedThresholdClassifierCV
from sklearn.model_selection import FixedThresholdClassifier

from sklearn.neighbors import KNeighborsRegressor, KNeighborsClassifier
from sklearn.svm import SVR

from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.linear_model import Lars, OrthogonalMatchingPursuit, SGDRegressor

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

from sklearn.feature_selection import SequentialFeatureSelector, RFECV

from sklearn.metrics import accuracy_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.metrics import classification_report
from sklearn.metrics import mean_squared_error
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_absolute_percentage_error
from sklearn.metrics import mean_squared_log_error
from sklearn.metrics import make_scorer

print(sklearn.__version__)


1.7.2


### Область работы 2 (выполнение лучшей модели)

In [4]:
# Данный блок(и) в области 2 выполняется преподавателем
#
# В области находится одна, единственная, итоговая модель машинного обучения с однозначными, 
# зафиксированными параметрами
#
# В данной области категорически запрещается искать, выбирать, улучшать, оптимизировать, 
# тюниговать и т.д. модель машинного обучения

In [5]:
# Путь к тренировочному набору
path_train = 'train.csv' # содержит только имя файла, без имен папок
# Путь к тестовому набору
path_test  = 'test.csv' # содержит только имя файла, без имен папок

In [6]:
# Блок(и) обучения и поверки модели

In [7]:
df_train = pd.read_csv(path_train)
df_test = pd.read_csv(path_test)

In [8]:
y = np.array(df_train.credit_risk)
X = df_train.drop(columns=['credit_risk','foreign_worker'])

X_test = df_test.drop(columns = ['foreign_worker'])

In [9]:
num_features = ['age', 'duration', 'amount']
cat_features = ['status', 'credit_history', 'purpose', 'savings', 'installment_rate',
       'other_debtors', 'present_residence',
       'other_installment_plans', 'housing', 'number_credits', 'job',
       'people_liable', 'telephone']

In [10]:
num_transformer = Pipeline(steps=[
    ('power', PowerTransformer(method='yeo-johnson')),
    ('scaler', StandardScaler()),
    ('poly', PolynomialFeatures(
        degree=3,
        include_bias=False
    ))
])

cat_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

CT = ColumnTransformer([
    ("num_transformer", num_transformer, num_features),
    ("cat_transformer", cat_transformer, cat_features)
]).set_output(transform='pandas')

display(CT)
ct = CT.fit_transform(X)
pd.DataFrame(ct).head().T

,transformers,"[('num_transformer', ...), ('cat_transformer', ...)]"
,remainder,'drop'
,sparse_threshold,0.3
,n_jobs,None
,transformer_weights,None
,verbose,False
,verbose_feature_names_out,True
,force_int_remainder_cols,'deprecated'
,method,'yeo-johnson'
,standardize,True
,copy,True


,0,1,2,3,4
num_transformer__age,1.773658,-1.041177,-0.724245,-1.403653,0.939404
num_transformer__duration,-0.016883,0.887332,0.697874,-1.175888,-1.807188
num_transformer__amount,-0.862958,1.509482,1.587918,-0.866164,0.850357
num_transformer__age^2,3.145861,1.084049,0.524530,1.970242,0.882480
num_transformer__age duration,-0.029945,-0.923869,-0.505431,1.650539,-1.697680
num_transformer__age amount,-1.530593,-1.571637,-1.150041,1.215794,0.798828
num_transformer__duration^2,0.000285,0.787358,0.487028,1.382712,3.265929
num_transformer__duration amount,0.014569,1.339411,1.108166,1.018512,-1.536755
num_transformer__amount^2,0.744697,2.278536,2.521483,0.750240,0.723107
num_transformer__age^3,5.579681,-1.128686,-0.379888,-2.765537,0.829005


In [11]:
best_pipe = Pipeline([
    ('preprocesing', CT),
    ('ml', GradientBoostingClassifier(
        n_estimators=4000,       
        learning_rate=0.05,      
        max_depth=1,             
        min_samples_leaf=10,     
        min_samples_split=30,
        subsample=0.8,           
        random_state=42,
        validation_fraction=0.3,   
        n_iter_no_change=30,        
        tol=1e-4
    ))])

best_model = best_pipe.fit(X, y)


In [12]:
# Блок предсказания с использованием тестового набора

In [13]:
y_predict = best_model.predict(X_test)

In [14]:
# Название вектора предсказанных значений  y_predict полученого на основании тестового набора


In [15]:
y_predict

array([1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1,
       1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 1,
       0, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1,
       1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0,
       1, 1, 1, 0, 1, 1, 1, 1], dtype=int64)